In [1]:
import torch
import torch.nn as nn
import math
import torch.nn.functional as F
torch.manual_seed(1)

In [2]:
torch.__version__

'2.9.1+cu126'

## torch.gather()
### 方法解析：torch.gather(input, dim, index, *, sparse_grad=False, out=None)->Tensor
### 作用：沿着指定维度收集特定索引处的值
### input:源张量
### dim：收集的维度
### index：索引张量，形状要与output一致

In [3]:
'''
第一题：使用gather在dim=0上收集对角线元素
'''
tensor = torch.randn([3, 4])
print( tensor)
print("output:", tensor.gather(0, torch.arange(3).reshape(1,3)).squeeze())

tensor([[ 0.6614,  0.2669,  0.0617,  0.6213],
        [-0.4519, -0.1661, -1.5228,  0.3817],
        [-1.0276, -0.5631, -0.8923, -0.0583]])
output: tensor([ 0.6614, -0.1661, -0.8923])


In [4]:
torch.arange(3).reshape(1,3)

tensor([[0, 1, 2]])

In [5]:
'''
第二题：dim=1上收集元素，实现矩阵阵列重排
目标：[[2, 1, 3],
    [5, 4, 6],
    [8, 7, 9]]
'''
tensor = torch.tensor([
    [1, 2, 3],
    [4, 5, 6],
    [7, 8, 9]
])
tensor.gather(1, torch.tensor([[1,0,2],[1,0,2],[1,0,2],]))

tensor([[2, 1, 3],
        [5, 4, 6],
        [8, 7, 9]])

In [6]:
'''
题目3：高级索引
实现一个批量数据处理，从每个样本的不同位置收集特征
# 索引：每个样本要收集的位置 [2, 0, 3, 1] (seq_len=4)
# 目标输出形状：(3, 4, 5)，每个位置按索引收集
'''
batch_data = torch.randn(3, 4, 5)
batch_data

tensor([[[ 0.4391,  1.1712,  1.7674, -0.0954,  0.1394],
         [-1.5785, -0.3206, -0.2993, -0.7984,  0.3357],
         [ 0.2753,  1.7163, -0.0561,  0.9107, -1.3924],
         [ 2.6891, -1.8821, -0.7765,  2.0242, -0.0865]],

        [[ 0.0981, -1.2150,  0.7312,  1.1718,  2.4070],
         [ 0.2786,  0.2468,  1.1843, -0.7282,  1.1633],
         [-0.0091, -0.8425,  0.2152, -0.5242, -1.8034],
         [-1.3083,  0.4533,  1.1422,  0.2486, -1.7754]],

        [[ 1.1173,  0.2981,  0.1099, -0.6463, -1.4344],
         [-0.5008,  0.1716, -0.1600, -0.5047, -1.4746],
         [-0.3416, -0.3003, -1.0483, -0.4709,  0.2911],
         [ 1.9907, -0.9247, -0.9301,  1.4301,  0.4208]]])

In [7]:
index = torch.tensor([2, 0, 3, 1]).unsqueeze(0).expand(3,-1)
index

tensor([[2, 0, 3, 1],
        [2, 0, 3, 1],
        [2, 0, 3, 1]])

In [8]:
index.unsqueeze(-1).expand(-1, -1, 5)

tensor([[[2, 2, 2, 2, 2],
         [0, 0, 0, 0, 0],
         [3, 3, 3, 3, 3],
         [1, 1, 1, 1, 1]],

        [[2, 2, 2, 2, 2],
         [0, 0, 0, 0, 0],
         [3, 3, 3, 3, 3],
         [1, 1, 1, 1, 1]],

        [[2, 2, 2, 2, 2],
         [0, 0, 0, 0, 0],
         [3, 3, 3, 3, 3],
         [1, 1, 1, 1, 1]]])

In [9]:
batch_data.gather(1, index.unsqueeze(-1).expand(-1, -1, 5)).shape

torch.Size([3, 4, 5])

In [10]:
'''
题目4：实际应用 - 分类损失
模拟分类任务中从logits收集目标类别的分数
目标：收集每个样本对应target的分数 [2.1, 0.9, 2.2]
'''
logits = torch.tensor([[0.1, 2.1, -0.5, 1.8],  # 4个类别的分数
                       [1.2, -0.3, 0.7, 0.9],
                       [-0.4, 1.5, 2.2, 0.1]])
targets = torch.tensor([1, 3, 2])  # 真实类别索引

In [11]:
logits.gather(1, targets.unsqueeze(1))

tensor([[2.1000],
        [0.9000],
        [2.2000]])

In [12]:
'''
题目5：复杂场景 - 序列标注
在序列标注任务中收集特定位置的隐藏状态
entity_positions: 每个样本的实体位置 [[1, 3], [0, 4]]
目标：收集每个实体的隐藏状态，输出形状(2, 2, 3)
'''
hidden_states = torch.randn(2, 5, 3)

In [13]:
index = torch.tensor([[1, 3], [0, 4]])
hidden_states.gather(1, index.unsqueeze(-1).expand(2, 2, 3))

tensor([[[ 1.1120,  0.6155,  0.1938],
         [-0.6200, -1.4782, -1.1334]],

        [[ 1.1996, -0.3030, -1.7618],
         [-1.7943, -1.5208,  0.9196]]])

In [14]:
index.unsqueeze(-1).expand(2, 2,3)

tensor([[[1, 1, 1],
         [3, 3, 3]],

        [[0, 0, 0],
         [4, 4, 4]]])

In [15]:
hidden_states

tensor([[[ 1.3851, -0.8138, -0.9276],
         [ 1.1120,  0.6155,  0.1938],
         [-2.5832,  0.8539, -2.1021],
         [-0.6200, -1.4782, -1.1334],
         [-0.1010,  0.3434,  0.3539]],

        [[ 1.1996, -0.3030, -1.7618],
         [ 0.6348, -0.8044, -1.0371],
         [-1.0669,  0.5431,  0.6607],
         [ 2.2952,  0.6749,  1.7133],
         [-1.7943, -1.5208,  0.9196]]])

## 维度操作专项训练

In [16]:
'''
题目1：维度分解理解
请手动索引出以下位置的值：
a) 第0个样本，第1个通道，第2行，第3列
b) 第1个样本，第2个通道，第3行，第4列
'''
tensor_4d = torch.arange(120).reshape(2, 3, 4, 5)

In [17]:
tensor_4d[0, 1, 2, 3]

tensor(33)

In [18]:
tensor_4d[1, 2, 3, 4]

tensor(119)

In [19]:
'''
题目2：形状变换的等价性
任务：以下哪些变换是等价的？为什么？
1. x.reshape(2, 12)
2. x.view(2, 12)
3. x.reshape(3, 8)
4. x.permute(1, 0, 2).reshape(3, 8)
'''
x = torch.arange(24).reshape(2, 3, 4)

In [20]:
x.reshape(2, 12)

tensor([[ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11],
        [12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]])

In [21]:
x.view(2, 12)

tensor([[ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11],
        [12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23]])

In [22]:
x.reshape(3, 8)

tensor([[ 0,  1,  2,  3,  4,  5,  6,  7],
        [ 8,  9, 10, 11, 12, 13, 14, 15],
        [16, 17, 18, 19, 20, 21, 22, 23]])

In [23]:
x.permute(1, 0, 2).reshape(3, 8)

tensor([[ 0,  1,  2,  3, 12, 13, 14, 15],
        [ 4,  5,  6,  7, 16, 17, 18, 19],
        [ 8,  9, 10, 11, 20, 21, 22, 23]])

In [24]:
'''
题目3：维度增删的实际意义
任务1：添加通道维度

任务2：扩展为RGB三通道

任务3：展平
'''
images = torch.randn(4, 28, 28)

In [25]:
with_channel = images.unsqueeze(1)
with_channel.shape

torch.Size([4, 1, 28, 28])

In [26]:
RGB = with_channel.expand(-1, 3, -1, -1)
RGB.shape

torch.Size([4, 3, 28, 28])

In [27]:
flatten = images.view(4, -1)
flatten.shape

torch.Size([4, 784])

In [28]:
'''
题目4：维度交换的视觉化
'''
feature_map = torch.arange(120).reshape(2, 3, 4, 5)

In [29]:
feature_map.permute(0, 2, 3, 1).shape

torch.Size([2, 4, 5, 3])

In [30]:
feature_map.permute(0, 3, 1, 2).shape

torch.Size([2, 5, 3, 4])

In [31]:
'''
题目5：广播机制的维度匹配
'''
A = torch.randn(3, 1, 4, 1)
B = torch.randn(2, 1, 5)
C = torch.randn(3, 1, 4, 5)

In [32]:
# A + B不会直接广播
# 维度从右对齐: A(3,1,4,1), B(2,1,5)
# 最后两个维度: 1和5(可广播), 4和1(可广播)
# 但第三个维度: 1和2(可广播), 但A有第四维3，B需要unsqueeze第一维
# 最终形状: (3,2,4,5)
(A + B).shape

torch.Size([3, 2, 4, 5])

In [33]:
#A的维度1可以扩展为任意大小
(A + C).shape

torch.Size([3, 1, 4, 5])

In [34]:
#无法广播，广播从右向左对齐，5和4无法广播
x = torch.randn(3, 1, 4)
y = torch.randn(1, 5)
(x + y).shape

RuntimeError: The size of tensor a (4) must match the size of tensor b (5) at non-singleton dimension 2

In [ ]:
#解决方案,将y的形状编程(1, 5, 1)
(x + y.unsqueeze(-1)).shape

In [ ]:
'''
小项目：注意力机制实现
'''
def attention(query, key, value):
    d_k = query.size(-1)
    attention_score = torch.matmul(query, key.transpose(-2, -1))/d_k**0.5
    attention_weights = torch.softmax(attention_score, dim=-1)
    attention_value = torch.matmul(attention_weights, value)
    return attention_value

batch_size = 2
seq_len_q = 3
seq_len_k = 4
d_k = 5
d_v = 6

query = torch.randn(batch_size, seq_len_q, d_k)
key = torch.randn(batch_size, seq_len_k, d_k)
value = torch.randn(batch_size, seq_len_k, d_v)

output = attention(query, key, value)
print(f"输出形状: {output.shape}")  # (2, 3, 6)

## torch.contiguous()

In [ ]:
'''
题目1：理解连续性
什么是连续张量？
张量在内存中连续存储，访问时stride是规则的
什么操作会导致不连续？
transpose()、t()：转置操作
permute()：维度重排
narrow()、select()：切片操作
expand()：扩展操作
'''
x = torch.arange(24).reshape(2, 3, 4)
x.is_contiguous()

In [ ]:
x.transpose(0,1).is_contiguous()

In [ ]:
z = x.transpose(0, 1).view(-1)#不连续导致的报错

In [ ]:
'''
题目2：contiguous的实际应用
'''
tensor_4d = torch.randn(2, 3, 4, 5)
permuted = tensor_4d.permute(0, 2, 3, 1)

In [ ]:
#尝试直接reshape
permuted.reshape(2, -1)

In [ ]:
#contiguous再reshape
permuted.contiguous().reshape(2, -1)

In [ ]:
#对比性能
import time

# 创建大张量
large_tensor = torch.randn(1000, 1000)

# 测试连续和不连续张量的操作性能
# 不连续版本
start = time.time()
non_contiguous = large_tensor.t()
for _ in range(100):
    result = non_contiguous * 2
print(f"不连续张量耗时: {time.time() - start:.4f}s")

# 连续版本
start = time.time()
contiguous = large_tensor.t().contiguous()
for _ in range(100):
    result = contiguous * 2
print(f"连续张量耗时: {time.time() - start:.4f}s")

## torch.cat() 和 torch.stack() 方法学习
### torch.cat()
#### 假设有N个形状为(A, B, C)的张量
#### cat(dim=0) → (N*A, B, C)
#### cat(dim=1) → (A, N*B, C)
#### cat(dim=2) → (A, B, N*C)

### torch.stack()
#### 假设有N个形状为(A, B, C)的张量
#### stack(dim=0) → (N, A, B, C)
#### stack(dim=1) → (A, N, B, C)
#### stack(dim=2) → (A, B, N, C)
#### stack(dim=3) → (A, B, C, N)

In [ ]:
'''
题目1：基础理解
任务：
1. 使用cat在dim=0上连接A和B
2. 使用cat在dim=1上连接A和B
3. 使用stack在dim=0上堆叠A和B
4. 使用stack在dim=1上堆叠A和B
'''
A = torch.tensor([[1, 2], [3, 4]])
B = torch.tensor([[5, 6], [7, 8]])

In [ ]:
torch.cat((A, B), dim=0).shape

In [ ]:
torch.cat((A, B), dim=1).shape

In [ ]:
torch.stack((A, B), dim=0).shape

In [ ]:
torch.stack((A, B), dim=1).shape

In [ ]:
'''
题目2：批量数据组合
任务：
1. 将这三个特征矩阵在dim=0上连接
2. 将这三个特征矩阵堆叠
'''
feat1 = torch.randn(16, 64)
feat2 = torch.randn(16, 64)
feat3 = torch.randn(16, 64)

In [ ]:
torch.cat((feat1, feat2, feat3), dim=0).shape

In [ ]:
torch.stack((feat1, feat2,feat3), dim=0).shape

In [ ]:
'''
题目3：序列数据组合
任务：
1. 将h1, h2, h3在seq_len维度上连接
2. 将h1, h2, h3在batch维度上连接
3. 将h1, h2, h3堆叠
'''
h1 = torch.randn(2, 3, 5)  # (batch_size, seq_len, hidden_size)
h2 = torch.randn(2, 3, 5)
h3 = torch.randn(2, 3, 5)

In [ ]:
torch.cat((h1, h2, h3), dim=1).shape

In [ ]:
torch.cat((h1, h2, h3), dim=0).shape

In [ ]:
torch.stack((h1, h2, h3), dim=0).shape

In [ ]:
'''
题目4：实际应用 - 多头注意力
任务：
1. 将四个头的输出在最后一个维度上连接
2. 将四个头的输出堆叠，然后重新排列为(2,5,32)的形状
'''
head1 = torch.randn(2, 5, 8)
head2 = torch.randn(2, 5, 8)
head3 = torch.randn(2, 5, 8)
head4 = torch.randn(2, 5, 8)

In [ ]:
torch.cat((head1, head2, head3, head4), dim=-1).shape

In [ ]:
torch.stack((head1, head2, head3, head4), dim=2).reshape(2, 5, 32)

## torch.masked_fill() 和 torch.where()
### torch.masked_fill(mask, value)
#### 作用：用value填充张量中mask为True的位置。
#### mask：布尔张量，形状必须与当前张量可广播
#### value：填充的标量值
#### 注意：会修改原张量或返回新张量（取决于是否使用in-place操作）

### torch.where(condition, x, y)
#### 作用：根据条件选择元素。
#### condition：布尔张量，条件
#### x：condition为True时选择的元素
#### y：condition为False时选择的元素
#### 注意：x和y可以是标量或张量，形状需可广播

In [ ]:
'''
题目1：基础理解
任务：
1. 使用masked_fill将大于5的元素填充为-1
2. 使用where将大于5的元素置为-1，其余保持原样
3. 使用where将大于5的元素置为-1，小于等于5的元素置为0
'''
tensor = torch.tensor([[1, 2, 3], [4, 5, 6], [7, 8, 9]], dtype=torch.float32)
mask = tensor > 5

In [ ]:
tensor.masked_fill(mask, -1)

In [ ]:
torch.where(mask, -1, tensor)

In [ ]:
torch.where(mask, -1, 0)

In [ ]:
'''
题目2：序列掩码
任务：
1. 将填充位置用-100填充（常用于忽略损失计算）
2. 将填充位置用0填充，非填充位置用1填充（得到注意力掩码）
3. 使用where将填充位置的值设置为0，非填充位置保持原样
'''
sequences = torch.tensor([[1, 2, 3, 0, 0],
                          [4, 5, 0, 0, 0],
                          [6, 7, 8, 9, 0]])

padding_mask = sequences == 0

In [ ]:
sequences.masked_fill(padding_mask, -100)

In [ ]:
torch.where(padding_mask, 0, 1)

In [ ]:
torch.where(padding_mask, 0, sequences)

In [ ]:
'''
题目3：梯度掩码
任务：
1. 使用masked_fill将不需要更新的行的梯度置零（注意：在反向传播之前如何操作？）
提示：可以使用register_hook或者直接操作grad属性
2. 使用where创建一个新的张量，该张量只包含需要更新的行，其余行为0
'''
weights = torch.randn(5, 3, requires_grad=True)

# 创建一个掩码，标记哪些行需要更新（假设第0, 2, 4行需要更新）
update_mask = torch.tensor([1, 0, 1, 0, 1], dtype=torch.bool)  # 形状(5,)

In [ ]:
#方法1：register hook
def gradient_mask_hook(grad):
    mask_expanded = update_mask.unsqueeze(-1).expand_as(grad)
    return grad*mask_expanded.float()

hook_handle = weights.register_hook(gradient_mask_hook)
outputs = weights.sum()
outputs.backward()
print(f"\n梯度更新掩码形状: {update_mask.shape}")
print(f"梯度掩码:\n{update_mask}")
print(f"\n原始梯度:\n{weights.grad}")


In [ ]:
#方式2：直接操作 grad 属性
weights = torch.randn(5, 3, requires_grad=True)
loss = (weights ** 2).sum()
loss.backward()
weights.grad.masked_fill(update_mask.unsqueeze(-1), 0)

In [ ]:
#方式3：使用 torch.where 创建新张量
weights = torch.randn(5, 3, requires_grad=True)
active_rows = torch.where(update_mask.unsqueeze(-1), weights,0)
active_rows

In [ ]:
'''
题目4：条件赋值
任务：
1. 比较两个预测，选择每个位置较大的值组成新的张量
2. 如果pred1的值大于0.5，则保留pred1的值，否则使用pred2的值
3. 将pred1和pred2中大于0.5的值置为1，小于等于0.5的值置为0
'''
pred1 = torch.tensor([[0.9, 0.1, 0.8],
                      [0.3, 0.7, 0.2]])
pred2 = torch.tensor([[0.6, 0.3, 0.7],
                      [0.4, 0.5, 0.9]])

In [ ]:
compare = pred1 > pred2
torch.where(compare, pred1, pred2)

In [ ]:
torch.where(pred1>0.5, pred1, pred2)

In [ ]:
torch.where(pred1 > 0.5,1,0)

In [ ]:
torch.where(pred2 > 0.5,1,0)

In [ ]:
'''
题目5：实际应用 - 注意力掩码
任务：
1. 使用seq_lens生成padding_mask（形状(3,5)）
2. 生成因果掩码（causal mask）掩盖未来位置（形状(5,5)）
3. 将padding_mask和causal_mask结合，得到最终的注意力掩码
注意：padding_mask需要广播，causal_mask对于每个样本是相同的
'''
seq_lens = torch.tensor([4, 3, 5])
max_len = 5

In [ ]:
batch_size = len(seq_lens)
position_ids = torch.arange(max_len).unsqueeze(0).expand(batch_size, max_len)#位置编码
padding_mask = position_ids < seq_lens.unsqueeze(1)
causal_mask = torch.tril(torch.ones(max_len, max_len)).bool()

padding_mask_expanded = padding_mask.unsqueeze(1)
causal_mask_expanded = causal_mask.unsqueeze(0)
combined_mask = padding_mask_expanded & causal_mask_expanded
print(combined_mask)
attention_score_mask = torch.where(combined_mask, 0, 1e-9)
print(attention_score_mask)

## torch.matmul() 和 torch.einsum()
### torch.matmul(input, other, *, out=None) → Tensor
#### 作用：矩阵乘积，支持广播
#### 对于2D张量：标准矩阵乘法
#### 对于高维张量：批量矩阵乘法
#### 支持自动广播
#### 比@操作符更灵活
#### 行为规则：
#### 1. 两个1D张量 → 点积（标量）
#### 2. 一个1D和一个2D → 向量-矩阵乘法
#### 3. 两个2D张量 → 标准矩阵乘法
#### 4. 两个ND张量 → 批量矩阵乘法（最后两维进行矩阵乘法）

### torch.einsum(equation, *operands) → Tensor
#### 作用：爱因斯坦求和约定，灵活的维度操作
#### equation：描述操作的字符串，如'ij,jk->ik'
#### *operands：输入张量
#### 优势：可以表达复杂的张量操作，代码简洁
#### 常用模式：
#### 'ij,jk->ik'：矩阵乘法
#### 'bij,bjk->bik'：批量矩阵乘法
#### 'ii->i'：取对角线
#### 'ijk->kji'：维度转置

In [ ]:
'''
题目1：基础矩阵操作对比
任务：
1. 使用matmul计算A×B
2. 使用einsum计算A×B
3. 使用@运算符计算A×B
4. 使用mm计算A×B（注意：mm不支持广播）
'''
A = torch.tensor([[1, 2], [3, 4]], dtype=torch.float32)
B = torch.tensor([[5, 6], [7, 8]], dtype=torch.float32)

In [ ]:
torch.matmul(A, B)

In [ ]:
torch.einsum('ij,jk->ik', A, B)

In [ ]:
A@B

In [ ]:
torch.mm(A, B)

In [ ]:
'''
题目2：批量矩阵乘法
任务：
1. 使用matmul进行批量矩阵乘法
2. 使用einsum进行批量矩阵乘法
3. 使用bmm进行批量矩阵乘法（注意：bmm要求三维）
'''
batch_size = 3
m, n, p = 4, 5, 6

A = torch.randn(batch_size, m, n)  # (3, 4, 5)
B = torch.randn(batch_size, n, p)  # (3, 5, 6)

In [ ]:
torch.matmul(A, B)

In [ ]:
torch.einsum('bij, bjk->bik',A, B)

In [ ]:
torch.bmm(A, B)

In [ ]:
'''
题目3：复杂张量操作
任务：
1. 使用matmul在最后两维进行矩阵乘法
2. 使用einsum实现相同的操作
3. 扩展：使用einsum计算A的转置与B的点积
4. 扩展：使用einsum计算A和B的批量对角线元素
'''
A = torch.randn(2, 3, 4, 5)  # (batch, seq, hidden1, hidden2)
B = torch.randn(2, 3, 5, 6)  # (batch, seq, hidden2, hidden3)

In [ ]:
torch.matmul(A,B).shape

In [ ]:
torch.einsum('bsij,bsjk->bsik', A, B).shape

In [ ]:
torch.einsum('bsji,bsjk->bsik', A.transpose(-1, -2), B).shape

In [ ]:
C = torch.randn(2, 3, 5, 5)
torch.einsum('bsjj->bsj', C).shape

In [ ]:
'''
题目4：实际应用 - 注意力机制
任务：
1. 使用matmul计算Q和K的转置的点积
2. 使用einsum实现相同的计算
3. 使用einsum一次性计算注意力权重和加权和
'''
batch_size = 2
seq_len = 3
d_k = 4
d_v = 5

Q = torch.randn(batch_size, seq_len, d_k)  # Query
K = torch.randn(batch_size, seq_len, d_k)  # Key
V = torch.randn(batch_size, seq_len, d_v)  # Value

In [ ]:
torch.matmul(Q, K.transpose(-2, -1)).shape

In [ ]:
torch.einsum('bqd, bkd->bqk', Q, K).shape

In [ ]:
def attention_einsum(Q, K, V):
    """使用einsum一次性计算注意力"""
    # 计算注意力分数
    d_k = Q.size(-1)
    scores = torch.einsum('bqd,bkd->bqk', Q, K) / (d_k ** 0.5)

    # softmax获取注意力权重
    attn_weights = torch.softmax(scores, dim=-1)

    # 加权求和
    output = torch.einsum('bqk,bkd->bqd', attn_weights, V)

    return output, attn_weights
output_einsum, weights_einsum = attention_einsum(Q, K, V)

In [ ]:
'''
题目5：张量缩并
任务：
1. 使用einsum计算 A_ijk * B_klm -> C_ijlm 的缩并
2. 使用einsum计算 A_ijk * B_klm * C_imn -> D_jln 的复杂缩并
3. 尝试用多个matmul操作实现第一个任务，比较代码复杂度
'''
A = torch.randn(3, 4, 5)
B = torch.randn(5, 6, 7)
C = torch.randn(3, 7, 8)

In [ ]:
torch.einsum('jik,klm->ijlm',A,B).shape

In [ ]:
torch.einsum('ijk,klm,imn->jln',A,B,C).shape

### 卷积神经网络
### nn.Conv2d
### H_out = (H+2P-K)/S +1

In [35]:
x = torch.randn(2, 3, 32, 32)
net = nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1)
net(x).shape

torch.Size([2, 16, 32, 32])

In [36]:
'''
in_channels = 3
out_channels = 64
kernel = 3×3
bias = True（默认）

问：总参数量多少？
答：1792

计算方法：out_channels × in_channels × k × k+bias(out_channels)

'''

'\nin_channels = 3\nout_channels = 64\nkernel = 3×3\nbias = True（默认）\n\n问：总参数量多少？\n答：1792\n\n计算方法：out_channels × in_channels × k × k+bias(out_channels)\n\n'

In [37]:
class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=8, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(8*16*16,10)
        )

    def forward(self,x):
        return self.net(x)

In [39]:
x = torch.randn(1,3,32,32)
x.shape

torch.Size([1, 3, 32, 32])

In [42]:
model = TinyCNN()
features = []
def hook_fn(module, input, output):
    features.append(output)

model.net[0].register_forward_hook(hook_fn)
x = torch.randn(1,3,32,32)
model(x)

print(features[0].shape)

torch.Size([1, 8, 32, 32])
